# 모듈 1: 투자 유니버스와 데이터

백테스트 기간의 각 시점마다 **그 시점에 실제로 투자 가능했던 종목**으로 유니버스를 구성하고, 이후 모듈이 공통으로 사용할 데이터를 만든다.

| 항목 | 선택 | 근거 |
|---|---|---|
| 투자 유니버스 | KOSPI 보통주 중 필터를 통과한 종목의 시가총액 상위 200개, 매월 말 재산정 | KOSPI200 벤치마크와 비교 가능한 규모, 과제 요건(최소 30종목) 충족 |
| 분석 기간 | 데이터 적재 2018.01 ~ 2025.12, 백테스트 2020.01 ~ 2025.12 | 과제 권장 기간. 2018 ~ 2019년은 상장기간 필터와 12개월 지표 계산용 준비 기간 |
| 종목 데이터 | KRX 일별 전종목 시세 (FinanceData `marcap` 저장소) | 상장폐지 종목까지 날짜별로 남아 있는 시점별 데이터 |
| 벤치마크 | KOSPI200 지수, KODEX 200 (`yfinance`) | 과제 권장 벤치마크 |
| 생존편향 | 시점별 유니버스 + 기간 중 퇴출 종목 포함 | Elton, Gruber & Blake (1996) |
| 전진편향 | t일 종가까지의 정보로 유니버스를 정하고 t+1일부터 적용 | Harvey & Liu (2015) |

**산출물** (`data/processed/`): `universe_monthly` (월별 유니버스), `kospi_daily` (편입 이력 종목의 일별 시세, 퇴출 종목 포함), `security_master` (종목별 거래 기간), `filter_log` (필터 단계별 종목 수), `benchmark_daily` (벤치마크)

## 1. 환경 설정

In [2]:
# 필요한 라이브러리 설치 (처음 한 번만 실행)
# pyarrow  : parquet 파일 읽기/쓰기
# yfinance : 벤치마크(지수, ETF) 데이터 수집
%pip install -q pandas numpy pyarrow requests yfinance


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import yfinance as yf

# ===== 분석 기간 =====
DATA_START = "2018-01-01"      # 원천 데이터 적재 시작일
                               # 백테스트 시작(2020.01) 시점에 2년치 이력이 있어야
                               # 상장기간 필터(252거래일)와 12개월 지표를 계산할 수 있음
BACKTEST_START = "2020-01-01"  # 백테스트 시작일 (과제 권장: 2020.01)
BACKTEST_END = "2025-12-31"    # 백테스트 종료일 (과제 권장: 2025.12)

# ===== 유니버스 필터 기준값 =====
# 기준값은 백테스트 결과를 보기 전에 고정한다.
# 결과를 보고 기준값을 바꾸면 여러 전략을 시험해 좋은 것만 고르는 셈이 되어
# 과적합 문제가 생긴다 (Harvey & Liu, 2015). 기준값 민감도는 모듈 9에서 점검한다.
MARKET = "KOSPI"          # 대상 시장
UNIVERSE_SIZE = 200       # [필터 5] 시가총액 상위 몇 개 종목을 유니버스로 쓸지
MIN_LISTING_DAYS = 252    # [필터 2] 최소 상장(거래) 일수: 약 1년
HALT_WINDOW = 20          # [필터 3] 최근 몇 거래일 안에 거래정지가 있었는지 확인할 기간
LIQ_WINDOW = 60           # [필터 4] 평균 거래대금 계산 기간: 약 3개월
LIQ_CUTOFF = 0.20         # [필터 4] 평균 거래대금 하위 몇 %를 제외할지

# [필터 1] 일반 기업이 아닌 상장 증권을 종목명으로 식별하는 패턴
#   리츠(부동산투자회사), 코크렙(구조조정 리츠), 인프라 펀드(맥쿼리인프라 등),
#   선박·광물 투자회사(바다로19호, 하이골드12호 등), 스팩(기업인수목적회사)
SPECIAL_NAME_PATTERN = r"(?:리츠|코크렙|인프라|\d+호)$|스팩"

# ===== 경로 =====
RAW_DIR = Path("data/raw")          # 원천 데이터 캐시 (한 번 받으면 재사용)
OUT_DIR = Path("data/processed")    # 산출물 저장 위치
RAW_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# KRX 일별 전종목 시세 (연도별 parquet 파일)
MARCAP_URL = "https://raw.githubusercontent.com/FinanceData/marcap/master/data/marcap-{year}.parquet"

## 2. 원천 데이터 적재

KRX가 매일 공개하는 전종목 시세를 연도별로 받는다. 각 행은 "특정 날짜에 거래된 한 종목"이므로 상장폐지된 종목도 폐지 전까지의 기록이 남아 있다. (국내 거래 기록 수집)

**일별 수익률.** `Changes`(전일 대비 등락폭)는 액면분할 등이 반영된 전일 기준가 대비 값이므로, `Close / (Close - Changes) - 1`로 계산하면 분할 이벤트에서도 수익률이 왜곡되지 않는다.

In [4]:
def load_krx_daily(start: str, end: str) -> pd.DataFrame:
    """
    KRX 일별 전종목 시세를 연도별로 받아 하나의 DataFrame으로 합침.
    이미 받은 연도 파일은 data/raw 폴더의 캐시를 재사용함.
    """
    frames = []
    for year in range(pd.Timestamp(start).year, pd.Timestamp(end).year + 1):
        path = RAW_DIR / f"marcap-{year}.parquet"
        if not path.exists():
            print(f"{year}년 데이터 다운로드 중...")
            res = requests.get(MARCAP_URL.format(year=year), timeout=180)
            res.raise_for_status()           # 다운로드 실패 시 여기서 오류를 냄
            path.write_bytes(res.content)
        frames.append(pd.read_parquet(path))

    df = pd.concat(frames, ignore_index=True)
    df["Date"] = pd.to_datetime(df["Date"])
    return df[(df["Date"] >= start) & (df["Date"] <= end)]


raw = load_krx_daily(DATA_START, BACKTEST_END)

# 대상 시장만 남김. 시장 구분은 '그날의' 소속 시장 기준이라,
# 코스닥에서 코스피로 이전 상장한 종목은 이전한 날부터 자연스럽게 포함됨
kospi = raw.loc[raw["Market"] == MARKET, [
    "Date", "Code", "Name",
    "Open", "High", "Low", "Close", "Changes",
    "Volume",     # 거래량 (주)
    "Amount",     # 거래대금 (원)
    "Marcap",     # 시가총액 (원)
    "Stocks",     # 상장주식수
]].sort_values(["Date", "Code"]).reset_index(drop=True)
del raw   # 메모리 절약

# 일별 수익률: 전일 기준가(= 종가 - 등락폭) 대비 종가 변화율
base_price = kospi["Close"] - kospi["Changes"]
kospi["ret"] = np.where(base_price > 0, kospi["Close"] / base_price - 1, np.nan)

print(f"기간        : {kospi['Date'].min().date()} ~ {kospi['Date'].max().date()}")
print(f"거래일 수    : {kospi['Date'].nunique():,}일")
print(f"종목 수(누적): {kospi['Code'].nunique():,}개 (기간 중 한 번이라도 {MARKET}에서 거래된 종목)")
kospi.head()

기간        : 2018-01-02 ~ 2025-12-30
거래일 수    : 1,963일
종목 수(누적): 1,032개 (기간 중 한 번이라도 KOSPI에서 거래된 종목)


,Date,Code,Name,Open,High,Low,Close,Changes,Volume,Amount,Marcap,Stocks,ret
0,2018-01-02,000020,동화약품,9750.0,9900.0,9700.0,9870.0,120.0,120676.0,1.181357e+09,2.756836e+11,27931470,0.012308
1,2018-01-02,000030,우리은행,15900.0,16000.0,15750.0,15900.0,150.0,653628.0,1.034937e+10,1.074840e+13,676000000,0.009524
2,2018-01-02,000040,KR모터스,0.0,0.0,0.0,325.0,0.0,0.0,0.000000e+00,6.118629e+10,188265520,0.000000
3,2018-01-02,000050,경방,13950.0,14100.0,13750.0,13850.0,-50.0,28945.0,4.013302e+08,3.797015e+11,27415270,-0.003597
4,2018-01-02,000060,메리츠화재,23450.0,23650.0,23350.0,23350.0,-150.0,120794.0,2.831719e+09,2.576392e+12,110338000,-0.006383


## 3. 유니버스 필터

매월 마지막 거래일(t)에 아래 5단계 필터를 순서대로 적용하고, 통과한 종목을 다음 거래일(t+1)부터 한 달간 사용한다. 필터 값은 모두 **t일 종가까지 공개된 정보**로 계산한다.

| 단계 | 기준 | 근거 |
|---|---|---|
| 1. 종목 유형 | 보통주만. 우선주, 리츠, 인프라·선박 펀드, 스팩 제외 | 우선주는 보통주와 중복이고, 리츠·펀드·스팩은 일반 기업과 수익 구조가 달라 팩터 비교가 성립하지 않음 |
| 2. 상장 기간 | 252거래일(약 1년) 이상 거래 이력 | 12-1 모멘텀, 252일 변동성 계산에 1년치 가격 필요. 상장 직후 종목은 가격 변동이 비정상적으로 큼 |
| 3. 거래 가능성 | 최근 20거래일 중 거래정지(거래량 0) 이력 없음 | 매매 불가능 종목 제외. 관리종목 지정이나 상장폐지 심사의 신호인 경우가 많음 |
| 4. 유동성 | 최근 60거래일 평균 거래대금 하위 20% 제외 | Korajczyk & Sadka (2004): 거래비용을 반영하면 유동성이 낮은 종목에서 전략 수익이 크게 줄어듦. 상대 순위 기준이라 시장 거래대금이 변해도 일관됨 |
| 5. 규모 | 남은 종목 중 시가총액 상위 200개 | KOSPI200과 비교 가능한 대형주 중심. 초소형주는 거래비용과 가격 왜곡이 큼 |

In [5]:
def compute_features(df: pd.DataFrame) -> dict:
    """
    필터에 필요한 지표를 '날짜 x 종목' 표로 미리 계산.
    모든 지표는 해당 날짜와 그 이전 데이터만 사용함 (rolling, cumsum은 과거 방향으로만 계산됨).
    """
    amount = df.pivot(index="Date", columns="Code", values="Amount")   # 거래대금
    volume = df.pivot(index="Date", columns="Code", values="Volume")   # 거래량

    return {
        # [필터 2] 누적 거래일 수: 그날까지 데이터가 존재한 날의 수
        "listing_days": amount.notna().cumsum(),
        # [필터 3] 최근 HALT_WINDOW일 중 거래량 0인 날의 수 (거래정지 일수)
        "halt_days": (volume == 0).astype(int).rolling(HALT_WINDOW, min_periods=1).sum(),
        # [필터 4] 최근 LIQ_WINDOW일 평균 거래대금 (60일치가 다 있어야 계산됨)
        "avg_amount": amount.rolling(LIQ_WINDOW, min_periods=LIQ_WINDOW).mean(),
    }


def build_snapshot(as_of: pd.Timestamp, day_rows: pd.DataFrame, feats: dict) -> tuple:
    """
    as_of(t일) 기준으로 5단계 필터를 적용해 유니버스를 만듦.
    day_rows : t일에 거래된 종목들의 행 (t일에 상장돼 있던 종목만 들어 있음)
    반환값   : (유니버스 DataFrame, 단계별 종목 수 기록 dict)
    """
    d = day_rows.set_index("Code")
    log = {"as_of_date": as_of, "n_listed": len(d)}

    # [필터 1] 종목 유형: 보통주(종목코드 끝자리 0)이면서 특수 증권이 아닌 것
    #   한국 종목코드는 보통주 끝자리가 0, 우선주는 5, 7, 9, K 등
    is_common = d.index.str.endswith("0")
    is_special = d["Name"].str.contains(SPECIAL_NAME_PATTERN, regex=True)
    d = d[is_common & ~is_special]
    log["n_type"] = len(d)

    # [필터 2] 상장 기간: t일까지 누적 거래일이 MIN_LISTING_DAYS 이상
    d = d.assign(listing_days=feats["listing_days"].loc[as_of, d.index].values)
    d = d[d["listing_days"] >= MIN_LISTING_DAYS]
    log["n_listing"] = len(d)

    # [필터 3] 거래 가능성: 최근 HALT_WINDOW일 동안 거래정지가 한 번도 없음
    halt = feats["halt_days"].loc[as_of, d.index].values
    d = d[halt == 0]
    log["n_tradable"] = len(d)

    # [필터 4] 유동성: 평균 거래대금이 '이 단계까지 남은 종목들' 중 하위 LIQ_CUTOFF 미만이면 제외
    d = d.assign(avg_amount=feats["avg_amount"].loc[as_of, d.index].values)
    d = d.dropna(subset=["avg_amount"])
    threshold = d["avg_amount"].quantile(LIQ_CUTOFF)
    d = d[d["avg_amount"] >= threshold]
    log["n_liquid"] = len(d)

    # [필터 5] 규모: t일 시가총액 상위 UNIVERSE_SIZE개
    d = d.sort_values("Marcap", ascending=False).head(UNIVERSE_SIZE)
    d["cap_rank"] = np.arange(1, len(d) + 1)
    log["n_universe"] = len(d)

    snapshot = d.reset_index()[["Code", "Name", "Close", "Marcap", "avg_amount", "listing_days", "cap_rank"]]
    return snapshot, log

In [6]:
# ===== 유니버스 산정일: 매월 마지막 거래일 =====
calendar = pd.DatetimeIndex(sorted(kospi["Date"].unique()))            # 전체 거래일 달력
month_ends = pd.Series(calendar, index=calendar).groupby(calendar.to_period("M")).max()

schedule = []
for as_of in month_ends:
    pos = calendar.get_loc(as_of)
    if pos + 1 >= len(calendar):
        continue                                   # 다음 거래일이 없는 마지막 날은 제외
    effective = calendar[pos + 1]                  # 적용일 = 산정일의 다음 거래일 (t+1)
    if pd.Timestamp(BACKTEST_START) <= effective <= pd.Timestamp(BACKTEST_END):
        schedule.append((as_of, effective))

# ===== 전체 기간 유니버스 산정 =====
features = compute_features(kospi)

snapshots, logs = [], []
for as_of, effective in schedule:
    day_rows = kospi[kospi["Date"] == as_of]                 # t일에 거래된 종목만
    snap, log = build_snapshot(as_of, day_rows, features)
    snap.insert(0, "effective_date", effective)
    snap.insert(0, "as_of_date", as_of)
    snapshots.append(snap)
    logs.append(log)

universe = pd.concat(snapshots, ignore_index=True)
filter_log = pd.DataFrame(logs)

print(f"유니버스 산정 {len(schedule)}회: {schedule[0][0].date()} ~ {schedule[-1][0].date()}")
print(f"한 번이라도 편입된 종목 수: {universe['Code'].nunique()}개")

# 필터 단계별 평균 종목 수
steps = {
    "n_listed": "상장 종목 전체",
    "n_type": "1. 종목 유형",
    "n_listing": "2. 상장 기간",
    "n_tradable": "3. 거래 가능성",
    "n_liquid": "4. 유동성",
    "n_universe": "5. 시가총액 상위",
}
summary = filter_log[list(steps)].agg(["mean", "min", "max"]).T.round(0).astype(int)
summary.index = summary.index.map(steps)
summary

유니버스 산정 72회: 2019-12-30 ~ 2025-11-28
한 번이라도 편입된 종목 수: 336개


,mean,min,max
상장 종목 전체,941,908,963
1. 종목 유형,803,779,824
2. 상장 기간,790,768,813
3. 거래 가능성,772,751,790
4. 유동성,618,601,632
5. 시가총액 상위,200,200,200


## 4. 생존편향 대응

오늘의 대형주 목록은 지난 몇 년을 잘 버틴 기업들이므로, 이 목록으로 과거를 테스트하면 실패한 기업을 처음부터 피한 결과가 나온다(Elton, Gruber & Blake, 1996). 이를 다음과 같이 막는다.

1. **시점별 유니버스:** 매월 그 시점의 데이터로 유니버스를 다시 산정한다(3절). 당시 대형주였다가 이후 밀려난 종목도 당시 유니버스에는 포함된다.
2. **퇴출 종목 포함:** 상장폐지·합병 등으로 사라진 종목의 거래 기록이 퇴출 직전까지 남아 있어, 보유 중 퇴출된 종목의 손실이 수익률에 반영된다.
3. **퇴출 이후 처리:** 마지막 거래일 종가로 청산해 현금(수익률 0%)으로 보유한 것으로 본다.

In [7]:
# ===== 종목 마스터: 편입 이력이 있는 종목의 거래 기간 =====
codes_ever = universe["Code"].unique()
prices = kospi[kospi["Code"].isin(codes_ever)].copy()   # 저장할 일별 시세 (퇴출 종목 포함)

master = prices.groupby("Code").agg(
    name=("Name", "last"),            # 마지막 거래일 기준 종목명
    first_date=("Date", "min"),       # 데이터 적재 시작(2018.01) 이후 첫 거래일
    last_date=("Date", "max"),        # 마지막 거래일
)
master["exited"] = master["last_date"] < calendar[-1]      # 기간 종료 전에 거래가 끝난 종목
master["months_in_universe"] = universe.groupby("Code").size()

exited = master[master["exited"]].sort_values("last_date")
print(f"편입 이력 종목 {len(master)}개 중 기간 내 거래 종료(상장폐지·합병 등): {len(exited)}개")
exited[["name", "last_date", "months_in_universe"]]

편입 이력 종목 336개 중 기간 내 거래 종료(상장폐지·합병 등): 9개


,name,last_date,months_in_universe
Code,,,
079440,오렌지라이프,2020-02-13,1
033660,우리금융캐피탈,2021-08-26,7
015350,부산가스,2022-01-17,1
000060,메리츠화재,2023-02-20,37
008560,메리츠증권,2023-04-24,40
003410,쌍용C&E,2024-07-08,52
115390,락앤락,2024-12-06,7
049770,동원F&B,2025-07-30,9
010620,HD현대미포,2025-12-12,71


## 5. 전진편향 대응

과거 시점의 의사결정에 그 시점에는 알 수 없었던 정보가 섞이지 않도록 다음 규칙을 적용했다(Harvey & Liu, 2015).

| 규칙 | 구현 |
|---|---|
| t일 정보로 결정하고 t+1일에 적용 | 유니버스는 월말(t) 종가 기준으로 산정하고 `effective_date`(t+1)부터 사용 |
| 과거 방향 계산만 사용 | 상장일수, 거래정지, 평균 거래대금 모두 t일 이전 값만 사용. 전체 기간 평균 같은 값은 쓰지 않음 |
| 수정주가를 가격 수준 판단에 쓰지 않음 | 수정주가는 이후의 분할·배당을 소급 반영한 값이라 미래 정보가 섞임. 필터에는 당시 실제 가격과 시가총액을 사용 |
| 현재 기준 종목 목록을 쓰지 않음 | 매월 그 시점에 상장돼 있던 종목만 대상으로 산정 |

In [8]:
# 모든 유니버스의 적용일이 산정일보다 뒤인지 확인 (t일 정보로 t+1일에 적용)
assert (universe["effective_date"] > universe["as_of_date"]).all()
print("확인: 모든 유니버스는 산정일 다음 거래일부터 적용됨")

확인: 모든 유니버스는 산정일 다음 거래일부터 적용됨


## 6. 벤치마크

과제 권장 벤치마크인 KOSPI200 지수를 `FinanceDataReader`로 수집한다.

| 열 | 기준 | 용도 |
|---|---|---|
| `KOSPI200` | 지수 종가 (가격수익률) | 유니버스 수익률이 배당 미반영 가격 기준이므로, 벤치마크도 같은 기준으로 비교 |

In [14]:
# 설치 (처음 한 번만): %pip install -q finance-datareader
import FinanceDataReader as fdr

# 벤치마크: KOSPI200 지수 (가격지수, 배당 미포함)
# yfinance의 ^KS200은 조회되지 않는 경우가 있어 FinanceDataReader로 수집함
# 유니버스 수익률도 배당 미반영 가격 기준이라 같은 기준으로 비교됨
BENCHMARK_CODE = "KS200"   # FinanceDataReader의 KOSPI200 지수 코드

try:
    df = fdr.DataReader(BENCHMARK_CODE, DATA_START, BACKTEST_END)
    benchmark = df[["Close"]].rename(columns={"Close": "KOSPI200"})
    benchmark.index = pd.to_datetime(benchmark.index).tz_localize(None)    # 시간대 정보 제거 (KRX 데이터와 날짜 맞추기)
    benchmark.index.name = "Date"
    if benchmark.dropna(how="all").empty:
        raise ValueError("빈 데이터")
    print(f"벤치마크 수집 완료 (KOSPI200): {benchmark.index.min().date()} ~ {benchmark.index.max().date()}")
except Exception as e:
    benchmark = None
    print(f"벤치마크 수집 실패 (FinanceDataReader 연결 확인 필요): {e}")

벤치마크 수집 완료 (KOSPI200): 2018-01-02 ~ 2025-12-30


## 7. 저장

이후 모듈은 유니버스를 반드시 `effective_date` 기준으로 결합한다(예: 2020년 1월 수익률에는 `effective_date`가 2020년 1월 첫 거래일인 유니버스를 사용). CSV는 엑셀 확인용이다.

In [10]:
universe.to_parquet(OUT_DIR / "universe_monthly.parquet", index=False)
universe.to_csv(OUT_DIR / "universe_monthly.csv", index=False, encoding="utf-8-sig")   # 엑셀 확인용

prices.to_parquet(OUT_DIR / "kospi_daily.parquet", index=False)
master.to_csv(OUT_DIR / "security_master.csv", encoding="utf-8-sig")
filter_log.to_csv(OUT_DIR / "filter_log.csv", index=False, encoding="utf-8-sig")
if benchmark is not None:
    benchmark.to_parquet(OUT_DIR / "benchmark_daily.parquet")

for p in sorted(OUT_DIR.iterdir()):
    print(f"{p.name:<28} {p.stat().st_size / 1e6:6.1f} MB")

benchmark_daily.parquet         0.0 MB
filter_log.csv                  0.0 MB
kospi_daily.parquet            19.3 MB
security_master.csv             0.0 MB
universe_monthly.csv            1.3 MB
universe_monthly.parquet        0.4 MB


In [12]:
# ===== 일별 시세 확인용 파일 저장 =====
# kospi_daily.parquet은 한 행이 '날짜 x 종목' 하나인 긴(long) 형태라 행이 수십만 개임
# 엑셀에서 보기 쉽도록 행 = 날짜, 열 = 종목명 인 넓은(wide) 표로 바꿔서 CSV로도 저장

# 종가 표: 행은 거래일, 열은 종목코드, 값은 그날 종가
close_wide = prices.pivot(index="Date", columns="Code", values="Close")

# 열 이름을 '종목명(코드)'로 바꿔서 엑셀에서 알아보기 쉽게 함
# (master에는 종목코드별 마지막 거래일 기준 종목명이 들어 있음)
close_wide.columns = [f"{master.loc[c, 'name']}({c})" for c in close_wide.columns]

# 수익률 표도 같은 형태로 만듦 (백테스트에서 실제로 쓰는 값)
ret_wide = prices.pivot(index="Date", columns="Code", values="ret")
ret_wide.columns = close_wide.columns

# utf-8-sig: 엑셀에서 한글 종목명이 깨지지 않게 하는 인코딩
close_wide.to_csv(OUT_DIR / "daily_close_wide.csv", encoding="utf-8-sig")
ret_wide.to_csv(OUT_DIR / "daily_return_wide.csv", encoding="utf-8-sig")

print(f"일별 종가 표: {close_wide.shape[0]}거래일 x {close_wide.shape[1]}종목")
# 상장 전이거나 상장폐지 이후인 날은 값이 비어 있음(NaN). 데이터 오류가 아님

# 노트북에서 바로 확인: 시가총액 상위 5개 종목의 최근 5거래일 종가
top5 = universe[universe["as_of_date"] == universe["as_of_date"].max()].nsmallest(5, "cap_rank")["Code"]
# reset_index(): 인덱스(Date)를 일반 열로 바꿔서 종목명과 같은 헤더 줄에 표시되게 함
close_wide[[f"{master.loc[c, 'name']}({c})" for c in top5]].tail().reset_index()

일별 종가 표: 1963거래일 x 336종목


,Date,삼성전자(005930),SK하이닉스(000660),LG에너지솔루션(373220),현대차(005380),두산에너빌리티(034020)
0,2025-12-23,111500.0,584000.0,388000.0,287000.0,77600.0
1,2025-12-24,111100.0,588000.0,390500.0,289000.0,75900.0
2,2025-12-26,117000.0,599000.0,383500.0,286000.0,73600.0
3,2025-12-29,119500.0,640000.0,380000.0,293500.0,76500.0
4,2025-12-30,119900.0,651000.0,368500.0,296500.0,75300.0
